# Forelesning 7: Numerisk integrasjon

I dag skal vi

- tilnærme et integral som en sum av små arealer: rektangler og trapeser
- undersøke hvor raskt metodene **konvergerer** når vi bruker flere intervaller
- bruke ferdige funksjoner i SciPy
- integrere topper i et kromatogram

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate

plt.rcParams.update({"font.size": 14, "figure.figsize": (8, 4.5)})

<div class="diskuter">

**Diskuter:** I et kromatogram er arealet under en topp et mål på stoffmengden. Hvordan finner vi et areal når vi bare har målepunkter og ingen formel for kurven?

</div>

## Del 1: Rektangelmetoden

<div class="viktig">

Vi deler intervallet $[a, b]$ i $n$ striper med bredde

$$h = \frac{b - a}{n}$$

og tilnærmer hver stripe med et rektangel. Integralet er summen av arealene: $\int_a^b f(x)\,dx \approx \sum f(x_k) \cdot h$

</div>

<div class="bilde-midt" style="width:50%">

![Rektangler og trapeser](bilder/integrasjonsmetoder.png)

</div>

### Venstretilnærming
Vi integrerer $f(x) = \cos x + 2$ fra 2 til 12. Den eksakte verdien er $\left[\sin x + 2x\right]_2^{12} \approx 18{,}554$.

In [ ]:
def f(x):
    return np.cos(x) + 2

a = 2
b = 12
n = 10

h = (b - a) / n
area = 0
x = a
for k in range(n):
    area = area + f(x) * h
    x = x + h

print(area)

<div class="oppgave">

**Prøv selv:** Vi pakker løkka inn i en funksjon med et ekstra argument `position`, som er 0 for venstre kant, 1 for høyre kant og 0,5 for midtpunktet. Fyll inn startverdien til `x`. Hvilken metode er best?

</div>

In [ ]:
def rectangle(f, a, b, n, position=0):
    h = (b - a) / n
    area = 0
    x = a + position * h
    for k in range(n):
        area = area + f(x) * h
        x = x + h
    return area

exact = (np.sin(12) + 2*12) - (np.sin(2) + 2*2)

print("Left:    ", rectangle(f, 2, 12, 10, 0))
print("Right:   ", rectangle(f, 2, 12, 10, 1))
print("Midpoint:", rectangle(f, 2, 12, 10, 0.5))
print("Exact:   ", exact)

<div class="diskuter">

**Diskuter:** Se på figuren. Hvorfor blir midtpunktstilnærmingen så mye bedre? Hvilke funksjoner gir den et eksakt svar for?

</div>

## Del 2: Konvergens

Et numerisk svar bør ikke vurderes ut fra ett enkelt valg av $n$. Vi dobler antall intervaller og ser hvordan feilen endrer seg.

<div class="diskuter">

**Tipp først:** Hva skjer med feilen når vi dobler $n$? Fyll inn feilen for midtpunktstilnærmingen, og sammenlikn med venstretilnærmingen.

</div>

In [ ]:
for n in [10, 20, 40, 80, 160]:
    error_left = abs(rectangle(f, 2, 12, n, 0) - exact)
    error_mid = abs(rectangle(f, 2, 12, n, 0.5) - exact)
    print(f"n = {n:4d}   left: {error_left:.2e}   midpoint: {error_mid:.2e}")

<div class="viktig">

Når vi dobler $n$, blir feilen omtrent

- **halvert** for venstre- og høyretilnærmingen (feilen er proporsjonal med $1/n$)
- **fjerdedelt** for midtpunktstilnærmingen (feilen er proporsjonal med $1/n^2$)

Kontroller alltid et numerisk integral ved å øke $n$ og se om svaret stabiliserer seg.

</div>

## Del 3: Trapesmetoden

Rektanglene har en vannrett topp. Trekker vi i stedet en rett linje mellom funksjonsverdiene i de to endepunktene, får vi et **trapes**.

<div class="viktig">

Arealet av ett trapes med bredde $h$:

$$A_k = \frac{f(x_k) + f(x_{k+1})}{2}\cdot h$$

</div>

<div class="oppgave">

**Prøv selv:** Kopier løkka fra `rectangle`, og endre den slik at den summerer trapeser i stedet for rektangler. Test på $f(x) = x^3$ fra 0 til 5, som har eksakt verdi 156,25. Legg deretter trapesmetoden til i konvergenstabellen for $\cos x + 2$.

</div>

In [ ]:
def trapezoid(f, a, b, n):
    h = (b - a) / n
    area = 0
    x = a
    for k in range(n):
        area = area + (f(x) + f(x + h)) / 2 * h
        x = x + h
    return area

def cube(x):
    return x**3

print(trapezoid(cube, 0, 5, 100))

for n in [10, 20, 40, 80, 160]:
    error_mid = abs(rectangle(f, 2, 12, n, 0.5) - exact)
    error_trap = abs(trapezoid(f, 2, 12, n) - exact)
    print(f"n = {n:4d}   midpoint: {error_mid:.2e}   trapezoid: {error_trap:.2e}")

<div class="diskuter">

**Diskuter:** Både midtpunkts- og trapesmetoden har feil proporsjonal med $1/n^2$, men for $\cos x + 2$ er feilen til trapesmetoden omtrent dobbelt så stor. Trapesmetoden virker likevel mer naturlig for måledata. Hvorfor?

</div>

<div class="viktig">

**Videre:** Rektangler bruker en **konstant**, trapeser en **rett linje** og **Simpsons metode** et **andregradspolynom** mellom punktene. Mer informasjon om formen gir vanligvis mindre feil for glatte funksjoner.

</div>

## Del 4: SciPy

<div class="viktig">

| Funksjon | Bruk |
|---|---|
| `integrate.trapezoid(y, x)` | måledata: arrayer med x- og y-verdier |
| `integrate.simpson(y, x=x)` | måledata, Simpsons metode |
| `integrate.quad(f, a, b)` | en funksjon vi kan evaluere hvor vi vil |

Eldre kode bruker navnene `trapz` og `simps`, som er fjernet i nyere SciPy.

</div>

In [ ]:
x = np.linspace(2, 12, 101)
y = f(x)

print("trapezoid:", integrate.trapezoid(y, x))
print("simpson:  ", integrate.simpson(y, x=x))
print("quad:     ", integrate.quad(f, 2, 12))

<div class="diskuter">

**Diskuter:** `quad` gir to tall. Hva er det andre? Hvorfor kan vi ikke bruke `quad` på et kromatogram?

</div>

## Del 5: Areal under kromatografiske topper

Vi lager et kunstig kromatogram med to topper. Signalet består nå av **målepunkter**, ikke en formel.

In [ ]:
time = np.linspace(0, 10, 501)   # min
peak_1 = 1.2 * np.exp(-0.5 * ((time - 3.0) / 0.35)**2)
peak_2 = 0.8 * np.exp(-0.5 * ((time - 6.5) / 0.50)**2)
signal = peak_1 + peak_2

plt.plot(time, signal)
plt.xlabel("Retention time (min)")
plt.ylabel("Signal (a.u.)")
plt.show()

<div class="oppgave">

**Prøv selv:** Hvilken topp tror du har størst areal? Bruk en boolsk maske til å velge ut tidsintervallet for hver topp, og integrer med `integrate.trapezoid`. Masken for topp 1 er gitt.

</div>

In [ ]:
mask_1 = (time >= 2.0) & (time <= 4.2)
mask_2 = (time >= 5.0) & (time <= 8.0)

area_1 = integrate.trapezoid(signal[mask_1], time[mask_1])
area_2 = integrate.trapezoid(signal[mask_2], time[mask_2])

print(f"Area peak 1: {area_1:.3f} a.u.·min")
print(f"Area peak 2: {area_2:.3f} a.u.·min")

<div class="viktig">

Topp 1 er 50 % høyere enn topp 2, men arealene er nesten like, fordi topp 2 er bredere. Stoffmengden henger sammen med **arealet**, ikke høyden.

Et integral får enheten til $y$ ganger enheten til $x$, her a.u.·min. En kalibrering knytter arealet til konsentrasjon.

</div>

<div class="diskuter">

**Diskuter:** Hva skjer med arealet hvis vi velger smalere integrasjonsgrenser? Hvilke problemer får vi hvis toppene overlapper, eller hvis grunnlinja ikke er null?

</div>

__Oppgaver__:
1. Integrer $f(x)=x^2-2x+4$ fra 2 til 8 med venstre-, høyre- og midtpunktstilnærming og trapesmetoden. Bruk først $n=10$ og deretter $n=100$, og sammenlikn med den analytiske verdien.
2. Plott den absolutte feilen som funksjon av $n$ for venstre-, midtpunkts- og trapesmetoden med logaritmiske akser. Hva forteller stigningen til linjene?
3. Legg svak tilfeldig støy til kromatogrammet med `rng.normal`. Hvor mye endres topparealene? Hvordan påvirkes de av integrasjonsgrensene?
4. Et kalorimeter registrerer varmeeffekt $P(t)$ i watt hvert sekund. Forklar hvorfor $\int P(t)\,dt$ gir varme, lag et kunstig datasett og integrer det. Hvilken enhet får svaret?
5. Når vil du bruke `quad`, og når vil du bruke `trapezoid`? Gi ett kjemisk eksempel på hver.
6. To NMR-signaler har arealene 3,02 og 1,01. Hva kan arealforholdet tyde på om det relative antallet hydrogenatomer?